Sentiment Analysis Comparison: Tokenization → Statistics → Semantics

This notebook compares three approaches to movie review sentiment
classification on a balanced 5,000-review sample of the IMDB dataset:
(1) a custom-built tokenizer with negation handling, compared against
NLTK's tokenizer, (2) a statistical baseline using TF-IDF features with
Logistic Regression, and (3) a semantic model using sentence embeddings
with the same classifier. The notebook includes accuracy/F1 comparisons,
a detailed error analysis of cases where the two models disagree, and a
final discussion of whether the semantic model's performance justifies
its added computational cost.

In [1]:
# Download and extract the IMDB dataset
!wget -q https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xzf aclImdb_v1.tar.gz
print("Done extracting")

Done extracting


In [2]:
import os
print(os.listdir('aclImdb'))
print(os.listdir('aclImdb/train'))

['README', 'imdbEr.txt', 'imdb.vocab', 'test', 'train']
['unsup', 'urls_unsup.txt', 'labeledBow.feat', 'neg', 'pos', 'unsupBow.feat', 'urls_pos.txt', 'urls_neg.txt']


In [3]:
import pandas as pd
import random

def load_reviews(base_path, sample_size_per_class=2500):
    data = []
    for label in ['pos', 'neg']:
        folder = os.path.join(base_path, label)
        files = os.listdir(folder)
        random.seed(42)  # so results are reproducible
        chosen = random.sample(files, min(sample_size_per_class, len(files)))
        for fname in chosen:
            with open(os.path.join(folder, fname), encoding='utf-8') as f:
                text = f.read()
            data.append({'text': text, 'label': 1 if label == 'pos' else 0})
    return pd.DataFrame(data)

# Load a balanced 5,000-review sample from the train folder
df = load_reviews('aclImdb/train', sample_size_per_class=2500)

# Shuffle so pos/neg aren't in blocks
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
print(df['label'].value_counts())
df.head()

(5000, 2)
label
1    2500
0    2500
Name: count, dtype: int64


,text,label
0,"Wow baby, this is indeed some fine Asian horro...",1
1,"This move is slow, plodding, cold, dark, and w...",0
2,The title of this obscure and (almost righteou...,0
3,this one is out there. Not much to say about i...,1
4,Many reviews here explain the story and charac...,1


In [4]:
import re

def strip_html(text):
    """Remove HTML tags like <br /> from review text"""
    return re.sub(r'<[^>]+>', ' ', text)

# Quick test
sample = "This movie was great!<br /><br />I loved it."
print(strip_html(sample))

This movie was great!  I loved it.


In [5]:
def tokenize(text):
    """
    Custom tokenizer:
    1. Strip HTML
    2. Lowercase
    3. Handle punctuation (split it off as separate tokens)
    4. Handle negation (tag words after 'not'/'never'/etc. until punctuation)
    """
    text = strip_html(text)
    text = text.lower()

    # Put spaces around punctuation so it splits off as its own token
    text = re.sub(r'([.,!?;:()"])', r' \1 ', text)

    # Split on whitespace
    raw_tokens = text.split()

    negation_words = {"not", "no", "never", "n't", "cannot", "cant", "don't", "didn't", "isn't", "wasn't"}
    punctuation = {".", ",", "!", "?", ";", ":", "(", ")", '"'}

    tokens = []
    negate = False
    for tok in raw_tokens:
        if tok in punctuation:
            negate = False  # negation "wears off" at punctuation
            tokens.append(tok)
        elif tok in negation_words:
            negate = True
            tokens.append(tok)
        elif negate:
            tokens.append(tok + "_NEG")
        else:
            tokens.append(tok)

    return tokens

# Test it
sample = "This movie was not good, but the acting wasn't bad either."
print(tokenize(sample))

['this', 'movie', 'was', 'not', 'good_NEG', ',', 'but', 'the', 'acting', "wasn't", 'bad_NEG', 'either_NEG', '.']


In [6]:
sample_review = df['text'][0]
print("ORIGINAL:\n", sample_review[:300], "...\n")
print("TOKENIZED:\n", tokenize(sample_review)[:40])

ORIGINAL:
 Wow baby, this is indeed some fine Asian horror/gore, and a crazy outlandish movie. This is a Japanese splatterfest that reminded me a little of Tetsuo, except in this case with all the blood and guts, there is a bizarre love story. It's hard to imagine how they even dreamed up this visually stunnin ...

TOKENIZED:
 ['wow', 'baby', ',', 'this', 'is', 'indeed', 'some', 'fine', 'asian', 'horror/gore', ',', 'and', 'a', 'crazy', 'outlandish', 'movie', '.', 'this', 'is', 'a', 'japanese', 'splatterfest', 'that', 'reminded', 'me', 'a', 'little', 'of', 'tetsuo', ',', 'except', 'in', 'this', 'case', 'with', 'all', 'the', 'blood', 'and', 'guts']


In [7]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

nltk_tokens = word_tokenize(strip_html(sample_review).lower())
my_tokens = tokenize(sample_review)

print("MY TOKENIZER (first 40):\n", my_tokens[:40])
print("\nNLTK TOKENIZER (first 40):\n", nltk_tokens[:40])

MY TOKENIZER (first 40):
 ['wow', 'baby', ',', 'this', 'is', 'indeed', 'some', 'fine', 'asian', 'horror/gore', ',', 'and', 'a', 'crazy', 'outlandish', 'movie', '.', 'this', 'is', 'a', 'japanese', 'splatterfest', 'that', 'reminded', 'me', 'a', 'little', 'of', 'tetsuo', ',', 'except', 'in', 'this', 'case', 'with', 'all', 'the', 'blood', 'and', 'guts']

NLTK TOKENIZER (first 40):
 ['wow', 'baby', ',', 'this', 'is', 'indeed', 'some', 'fine', 'asian', 'horror/gore', ',', 'and', 'a', 'crazy', 'outlandish', 'movie', '.', 'this', 'is', 'a', 'japanese', 'splatterfest', 'that', 'reminded', 'me', 'a', 'little', 'of', 'tetsuo', ',', 'except', 'in', 'this', 'case', 'with', 'all', 'the', 'blood', 'and', 'guts']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


**Tokenizer Comparison Notes:

1. My tokenizer tags words after negation cues (e.g. "not", "wasn't") with
_NEG until the next punctuation mark, so "good" after "not" is treated as
a different token than "good" on its own. NLTK's word_tokenize has no
concept of negation — it just returns the plain words with no tagging.

2. NLTK splits contractions like "wasn't" into two separate tokens
(was, n't), while my tokenizer keeps "wasn't" as a single token.**

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 4000
Test size: 1000


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Convert text into TF-IDF numeric features
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Logistic Regression
stat_model = LogisticRegression(max_iter=1000)
stat_model.fit(X_train_tfidf, y_train)

# Predict on held-out test set
stat_preds = stat_model.predict(X_test_tfidf)

# Metrics
stat_acc = accuracy_score(y_test, stat_preds)
stat_f1 = f1_score(y_test, stat_preds)

print("Statistical Model (TF-IDF + Logistic Regression)")
print("Accuracy:", stat_acc)
print("F1 Score:", stat_f1)

Statistical Model (TF-IDF + Logistic Regression)
Accuracy: 0.866
F1 Score: 0.8683693516699411


In [10]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded


In [11]:
import numpy as np

# Convert train and test text into embeddings
X_train_emb = embed_model.encode(X_train.tolist(), show_progress_bar=True)
X_test_emb = embed_model.encode(X_test.tolist(), show_progress_bar=True)

print(X_train_emb.shape)
print(X_test_emb.shape)

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(4000, 384)
(1000, 384)


In [12]:
sem_model = LogisticRegression(max_iter=1000)
sem_model.fit(X_train_emb, y_train)

sem_preds = sem_model.predict(X_test_emb)

sem_acc = accuracy_score(y_test, sem_preds)
sem_f1 = f1_score(y_test, sem_preds)

print("Semantic Model (Sentence Embeddings + Logistic Regression)")
print("Accuracy:", sem_acc)
print("F1 Score:", sem_f1)

Semantic Model (Sentence Embeddings + Logistic Regression)
Accuracy: 0.811
F1 Score: 0.8119402985074626


In [13]:
# Build a results table for the test set
results_df = pd.DataFrame({
    'text': X_test.values,
    'true_label': y_test.values,
    'stat_pred': stat_preds,
    'sem_pred': sem_preds
})

# Find where the two models disagree with each other
disagreements = results_df[results_df['stat_pred'] != results_df['sem_pred']]

print("Number of disagreements:", len(disagreements))
disagreements.head(10)

Number of disagreements: 171


,text,true_label,stat_pred,sem_pred
1,"I just watched ""The Last Wave"" in my school's ...",1,0,1
10,Just like last years event WWE New Years Revol...,0,1,0
23,"First of all, this movie isn't a complete disa...",0,0,1
26,I am very surprised to see such a high rating ...,0,0,1
32,This is a well made informative film in the ve...,0,1,0
37,"I actually like the original, and this film ha...",1,0,1
38,A group of people are invited to there high sc...,0,1,0
44,Astounding.....This may have been A poor attem...,1,1,0
55,My Super Ex Girlfriend turned out to be a plea...,1,1,0
62,This is a bit of a puzzle for a lot of the art...,1,1,0


In [14]:
# Set pandas to show full text instead of truncating
pd.set_option('display.max_colwidth', None)

# Look at 8 random disagreements so we can pick the most interesting ones
sample_disagreements = disagreements.sample(8, random_state=1)

for i, row in sample_disagreements.iterrows():
    print(f"--- Review index {i} ---")
    print("TRUE LABEL:", "Positive" if row['true_label'] == 1 else "Negative")
    print("Statistical model predicted:", "Positive" if row['stat_pred'] == 1 else "Negative")
    print("Semantic model predicted:", "Positive" if row['sem_pred'] == 1 else "Negative")
    print("TEXT:", row['text'][:600])
    print()

--- Review index 536 ---
TRUE LABEL: Negative
Statistical model predicted: Positive
Semantic model predicted: Negative
TEXT: I like silent films, but this was a little too moronic. As much as I wish I could say that it was worth the hour I stood up I can't. I don't think any version of the movie even comes close to the book. And don't try it out on kids, they might freak. And the lady who played Pollyanna, how old was she? 38? I know the labor laws were different back then... BUT COME ON PEOPLE.

--- Review index 673 ---
TRUE LABEL: Positive
Statistical model predicted: Positive
Semantic model predicted: Negative
TEXT: This movie is a great way for the series to finally end. Peter (the boy from Puppet Master III) is all grown up and is now the Puppet Master. Well, this girl comes to destroy the puppets and learn Toulon's secrets but instead she listens to the story about the puppets. Most of this movie is footage from Puppet Master II, Puppet Master III, Puppet Master 4, Puppet Master 

Error Analysis
Example 1 — Review #536
True label: Negative | Statistical model: Positive | Semantic model: Negative (correct)
The review opens with "I like silent films" and uses several positive-toned words ("wish," "worth"), but the overall meaning is negative — the reviewer says it wasn't worth their time. The statistical model likely over-weighted individual positive words like "like," missing that the surrounding sentence structure negates them. The semantic model correctly captured the overall negative tone because it reads meaning across the whole sentence rather than counting isolated words.
Example 2 — Review #673
True label: Positive | Statistical model: Positive (correct) | Semantic model: Negative
This review is mostly a plot summary (character names, movie titles, franchise history) rather than direct opinion words. It does contain some negative-sounding words used neutrally ("destroy," "sorry," "wouldn't let"). The semantic model likely picked up on this negative-toned vocabulary and misread the overall tone, while the statistical model correctly leaned on the small number of clearly positive words ("great way... to finally end").
Example 3 — Review #420
True label: Positive | Statistical model: Positive (correct) | Semantic model: Negative
The review is clearly positive ("Bravo!", "believable," "great opportunity"), but it also discusses the negative concept of stereotyping in film ("hampered," "completely ruined," "stereotyping"). The statistical model correctly weighted the strong positive words like "Bravo." The semantic model, which reads overall meaning, may have been pulled toward a negative reading by the discussion of stereotyping, even though that discussion wasn't about this specific film.
Example 4 — Review #957
True label: Negative | Statistical model: Negative (correct) | Semantic model: Positive
The review contains clearly negative critique words ("laziest," "punching-bags," "impossibly simple") but is embedded in a plot-heavy description of an action-western (train robbery, gold, chain-gang escape). The semantic model may have associated the exciting genre/action content with a positive tone, missing that the reviewer's actual evaluation was critical.
Example 5 — Review #648
True label: Negative | Statistical model: Negative (correct) | Semantic model: Positive
This review uses complex, wordy sentence structures ("leadenly written," "big disappointment") mixed in with mentions of well-known actors and historical figures (Yul Brynner, Pancho Villa). The statistical model correctly caught the explicit negative phrases. The semantic model may have been distracted by the descriptive, name-heavy content, diluting the negative signal across a longer, more complex sentence.
Overall pattern: the statistical model tends to succeed when reviews contain a small number of strong, explicit sentiment words, even in long or descriptive reviews. The semantic model tends to fail when negative or positive vocabulary appears in a neutral, descriptive context (plot summaries, genre discussion) rather than as direct evaluation — it reads overall tone across the passage, so unrelated negative/positive content can pull its prediction the wrong way.

Does the semantic model's improvement justify its cost?
In this experiment, the semantic model (sentence embeddings) did not outperform the simpler statistical model — it scored lower on both accuracy (0.811 vs 0.866) and F1 (0.812 vs 0.868), while taking significantly longer to run (embedding 5,000 reviews took several minutes versus seconds for TF-IDF) and requiring an extra library download. For this dataset size (5,000 balanced reviews) and this task (movie sentiment, which relies heavily on explicit sentiment vocabulary), the added computational cost of semantic embeddings is not justified — TF-IDF's direct word-counting approach is well-suited to catching strong, explicit sentiment language. Semantic embeddings would likely become more valuable on harder tasks involving sarcasm, subtlety, or paraphrased sentiment, or with a larger training set that lets the embeddings' richer representation be fully leveraged by the classifier.